# Загрузка данных

In [1]:
import pandas as pd

df = pd.read_csv("data/WA_Fn-UseC_-HR-Employee-Attrition.csv")
print(f"Shape: {df.shape}")
display(df.head(3))

Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


# Проверка пропусков

In [2]:
df.isna().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

# Преобразование типов данных

In [3]:
df["is_attrition"] = (df["Attrition"] == "Yes").astype(int)
df["is_overtime"] = (df["OverTime"] == "Yes").astype(int)

str_cols = df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
    df[col] = df[col].str.strip()

# Проверки логической корректности в данных

In [4]:
checks = {
    "duplicated_rows": df.duplicated().sum(),
    "duplicated_id": df["EmployeeNumber"].duplicated().sum(),
    "missing_values": df.isna().sum().sum(),
    "invalid_age": (~df["Age"].between(18, 90)).sum(),
    "non_positive_rates": (df[["DistanceFromHome", "DailyRate", "HourlyRate", "MonthlyIncome", "MonthlyRate", "PercentSalaryHike"]] <= 0).any(axis=1).sum(),
    "negative_tenure": (df[["NumCompaniesWorked", "TotalWorkingYears", "YearsAtCompany", "YearsInCurrentRole", "YearsSinceLastPromotion", "YearsWithCurrManager"]] < 0).any(axis=1).sum(),
    "company_tenure_gt_total": (df["YearsAtCompany"] > df["TotalWorkingYears"]).sum(),
    "promotion_gt_company_tenure": (df["YearsSinceLastPromotion"] > df["YearsAtCompany"]).sum(),
    "started_under_16": ((df["Age"] - df["TotalWorkingYears"]) < 16).sum()
}

display(pd.Series(checks, name="errors_count"))

duplicated_rows                0
duplicated_id                  0
missing_values                 0
invalid_age                    0
non_positive_rates             0
negative_tenure                0
company_tenure_gt_total        0
promotion_gt_company_tenure    0
started_under_16               0
Name: errors_count, dtype: int64

# Проверка уникальности

In [5]:
df.nunique().rename("unique_count")

Age                           43
Attrition                      2
BusinessTravel                 3
DailyRate                    886
Department                     3
DistanceFromHome              29
Education                      5
EducationField                 6
EmployeeCount                  1
EmployeeNumber              1470
EnvironmentSatisfaction        4
Gender                         2
HourlyRate                    71
JobInvolvement                 4
JobLevel                       5
JobRole                        9
JobSatisfaction                4
MaritalStatus                  3
MonthlyIncome               1349
MonthlyRate                 1427
NumCompaniesWorked            10
Over18                         1
OverTime                       2
PercentSalaryHike             15
PerformanceRating              2
RelationshipSatisfaction       4
StandardHours                  1
StockOptionLevel               4
TotalWorkingYears             40
TrainingTimesLastYear          7
WorkLifeBa

In [6]:
for col in df.columns:
    if df[col].nunique() < 10:
        print(col, df[col].unique())

Attrition <ArrowStringArray>
['Yes', 'No']
Length: 2, dtype: str
BusinessTravel <ArrowStringArray>
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Length: 3, dtype: str
Department <ArrowStringArray>
['Sales', 'Research & Development', 'Human Resources']
Length: 3, dtype: str
Education [2 1 4 3 5]
EducationField <ArrowStringArray>
[   'Life Sciences',            'Other',          'Medical',
        'Marketing', 'Technical Degree',  'Human Resources']
Length: 6, dtype: str
EmployeeCount [1]
EnvironmentSatisfaction [2 3 4 1]
Gender <ArrowStringArray>
['Female', 'Male']
Length: 2, dtype: str
JobInvolvement [3 2 4 1]
JobLevel [2 1 3 4 5]
JobRole <ArrowStringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: str
JobSatisfaction [4 2 3 1]
MaritalStatus

In [7]:
df_unique_counts = df.nunique().rename("unique_count")
non_unique_counts = df_unique_counts[df_unique_counts == 1].index.to_list()

# Отбор информативных полей 

In [8]:
df = df.drop(non_unique_counts, axis=1)

# Сохранение в parquet для оптимизации и учета типов данных

In [9]:
df.to_parquet("data/clean.parquet", index=False)